# Notebook 3: Train Logistic Regression and XGBoost
This notebook trains baseline Logistic Regression and XGBoost models on the ICU dataset.
Like Random Forest, these models don't naturally handle 3D sequence data `(N, 24, 37)`, so we will flatten the time dimension to `(N, 24*37)`.


In [ ]:
import os
try:
    from google.colab import drive
    # Only mount if the drive isn't already mounted to avoid errors
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/ICU-Patient-Deterioration/preprocessing_pipeline/output'
    print('Running in Google Colab. Data directory set to Google Drive.')
except ImportError:
    # If not in Colab (running locally)
    data_dir = 'preprocessing_pipeline/output'
    print('Running Locally. Data directory set to local folder.')


In [ ]:
import numpy as np
import os

# Load Data
print('Loading data...')
X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))

print(f'Original X_train shape: {X_train.shape}')


### Flatten 3D Sequences for Baseline Models
We reshape `(N, Sequence_Length, Features)` into `(N, Sequence_Length * Features)`.


In [ ]:
N_train, seq_len, n_features = X_train.shape
N_val = X_val.shape[0]

X_train_flat = X_train.reshape(N_train, seq_len * n_features)
X_val_flat = X_val.reshape(N_val, seq_len * n_features)

print(f'Flattened X_train shape: {X_train_flat.shape}')
print(f'Flattened X_val shape: {X_val_flat.shape}')


### Train Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Initialize and train Logistic Regression model
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
print('Training Logistic Regression...')
lr_model.fit(X_train_flat, y_train)
print('Training complete!')

In [ ]:
# Logistic Regression Predictions & Evaluation
y_val_prob_lr = lr_model.predict_proba(X_val_flat)[:, 1]
print('--- Logistic Regression Validation Performance ---')
print(f'ROC-AUC Score: {roc_auc_score(y_val, y_val_prob_lr):.4f}')
print(f'PR-AUC (Average Precision) Score: {average_precision_score(y_val, y_val_prob_lr):.4f}')

### Train XGBoost


In [ ]:
import xgboost as xgb

# Calculate scale_pos_weight to handle class imbalance for XGBoost
neg_count = len(y_train) - np.sum(y_train)
pos_count = np.sum(y_train)
scale_pos_weight = neg_count / pos_count

# Initialize and train XGBoost model
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
print('Training XGBoost...')
xgb_model.fit(X_train_flat, y_train)
print('Training complete!')

In [ ]:
# XGBoost Predictions & Evaluation
y_val_prob_xgb = xgb_model.predict_proba(X_val_flat)[:, 1]
print('--- XGBoost Validation Performance ---')
print(f'ROC-AUC Score: {roc_auc_score(y_val, y_val_prob_xgb):.4f}')
print(f'PR-AUC (Average Precision) Score: {average_precision_score(y_val, y_val_prob_xgb):.4f}')

### Patient Risk Stratification (Using XGBoost)
Categorizing the probabilities into low, moderate, high, and very high risk levels using the best performing model (typically XGBoost).

In [ ]:
import pandas as pd

# Create a DataFrame to view the final predictions and risk levels
results_df = pd.DataFrame({
    'True_Label': y_val,
    'Deterioration_Probability': y_val_prob_xgb
})

# Define risk categories based on probability thresholds
# You can adjust these thresholds based on clinical requirements
def categorize_risk(prob):
    if prob < 0.25:
        return 'Low Risk'
    elif prob < 0.50:
        return 'Moderate Risk'
    elif prob < 0.75:
        return 'High Risk'
    else:
        return 'Very High Risk'

results_df['Risk_Level'] = results_df['Deterioration_Probability'].apply(categorize_risk)

print('Sample of Patient Predictions:')
display(results_df[['True_Label', 'Deterioration_Probability', 'Risk_Level']].head(20))

print('\nRisk Level Distribution in Validation Set (XGBoost):')
print(results_df['Risk_Level'].value_counts())
